In [1]:
# =============================================================================
# Dataset 2: mind-wandering classification
# Comparison: raw EEG versus ICA-cleaned EEG
# =============================================================================

# Keep numerical-library thread use bounded when processing high-dimensional EEG.
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

from pathlib import Path
from dataclasses import dataclass
import warnings
import gc

import numpy as np
import pandas as pd
import mne
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import welch
from scipy.stats import kurtosis, skew

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold

# =============================================================================
# Runtime configuration and publication plot defaults
# =============================================================================

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")
sns.set_theme(style="whitegrid", context="notebook")

# ============================================================
# SHARED PLOT STYLE & COLOR PALETTE  (identical across datasets)
# ============================================================
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.5,
})

ARTIFACT_PALETTE = {
    "Raw":          "#1f77b4",
    "ICA_Cleaned":  "#ff7f0e",
}

CHANCE_LINE = {"color": "#2b2b2b", "linestyle": "--", "linewidth": 1.2}
YLIM_ACCURACY = (0, 1)


def add_chance_line(ax, y=0.5):
    ax.axhline(y, **CHANCE_LINE)


def apply_format_accuracy(ax, title, ylabel="Mean CV Accuracy"):
    ax.set_ylim(*YLIM_ACCURACY)
    ax.set_yticks(np.arange(0, 0.9, 0.1))
    add_chance_line(ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)


# =============================================================================
# Analysis configuration and deterministic dataset locations
# =============================================================================

DATASET_ROOT = Path("/kaggle/input/datasets/jvkrishwanth/mwdataset")
OUTDIR = Path("/kaggle/working/eeg_mw_results/dataset2_classification")
PLOT_DIR = OUTDIR / "plots"

OUTDIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = ["sub-01", "sub-02"]
SESSIONS = range(1, 12)

# The Kaggle dataset is split across three downloaded archive folders per
# participant. Channel metadata for every session is stored in the -3-001
# folder, while the BDF recordings are distributed as mapped below.
BDF_ARCHIVES = {
    "sub-01": {
        1: "sub-01-20260605T131512Z-3-002", 2: "sub-01-20260605T131512Z-3-003",
        3: "sub-01-20260605T131512Z-3-002", 4: "sub-01-20260605T131512Z-3-002",
        5: "sub-01-20260605T131512Z-3-002", 6: "sub-01-20260605T131512Z-3-001",
        7: "sub-01-20260605T131512Z-3-001", 8: "sub-01-20260605T131512Z-3-001",
        9: "sub-01-20260605T131512Z-3-001", 10: "sub-01-20260605T131512Z-3-003",
        11: "sub-01-20260605T131512Z-3-003",
    },
    "sub-02": {
        1: "sub-02-20260605T131514Z-3-003", 2: "sub-02-20260605T131514Z-3-002",
        3: "sub-02-20260605T131514Z-3-002", 4: "sub-02-20260605T131514Z-3-001",
        5: "sub-02-20260605T131514Z-3-001", 6: "sub-02-20260605T131514Z-3-002",
        7: "sub-02-20260605T131514Z-3-002", 8: "sub-02-20260605T131514Z-3-001",
        9: "sub-02-20260605T131514Z-3-001", 10: "sub-02-20260605T131514Z-3-001",
        11: "sub-02-20260605T131514Z-3-003",
    },
}

METADATA_ARCHIVES = {
    "sub-01": "sub-01-20260605T131512Z-3-001",
    "sub-02": "sub-02-20260605T131514Z-3-001",
}

# Recordings are resampled before filtering, epoching, and feature extraction.
TARGET_SFREQ = 256.0

# Event codes encoded in the BDF Status channel.
EVENT_TRIAL_START = 10
EVENT_MW_REPORT = 30
EVENT_START_COUNTING = 50

# The configured analysis window is identical for both artifact conditions.
WINDOW_NAME = "Analysis_Window"
WINDOW_DURATION = 5.0
MW_START_OFFSET = -5.0
FOCUS_START_OFFSET = 1.0

WINDOWS = {
    WINDOW_NAME: {
        "duration": WINDOW_DURATION,
        "mw_start_offset": MW_START_OFFSET,
        "focus_start_offset": FOCUS_START_OFFSET,
        "description": "MW -5:-0.1, Focus +1:6",
    },
}

ARTIFACT_CONDITIONS = ["Raw", "ICA_Cleaned"]

# Eighteen time-domain, Hjorth, spectral, and entropy features per EEG channel.
FEATURE_NAMES = [
    "Mean", "Variance", "Std", "RMS", "Kurtosis", "Skewness",
    "HjorthActivity", "HjorthMobility", "HjorthComplexity", "Peak2Peak",
    "TotalPower", "ThetaPower", "AlphaPower", "SpectralEntropy",
    "DominantFreq", "Energy", "ShannonEntropy", "DifferentialEntropy",
]


# =============================================================================
# Dataset access and preprocessing helpers
# =============================================================================

@dataclass
class SessionConfig:
    subject: str
    session: int
    bdf_path: Path
    channels_tsv: Path


def make_session_config(subject, session):
    ses = f"ses-{session}"
    eeg_dir = DATASET_ROOT / BDF_ARCHIVES[subject][session] / subject / "eeg"
    metadata_dir = DATASET_ROOT / METADATA_ARCHIVES[subject] / subject / "eeg"
    return SessionConfig(
        subject=subject,
        session=session,
        bdf_path=eeg_dir / f"{subject}_{ses}_task-BreathCounting_eeg.bdf",
        channels_tsv=metadata_dir / f"{subject}_{ses}_task-BreathCounting_channels.tsv",
    )


def deduplicate_events(events):
    # Resampling can map distinct original samples to the same target sample.
    if len(events) == 0:
        return events
    df = pd.DataFrame(events, columns=["sample", "previous", "code"])
    df = df.drop_duplicates(subset=["sample", "code"], keep="first")
    df = df.sort_values(["sample", "code"])
    return df[["sample", "previous", "code"]].to_numpy(dtype=int)


def get_eeg_exg_names(raw, channels_tsv):
    # EEG identities come from the BIDS channel table; EXG1-EXG8 are EXG reference channels.
    table = pd.read_csv(channels_tsv, sep="\t")
    eeg_names = (
        table.loc[table["channelTypes"].fillna("").str.upper() == "EEG", "name"]
        .astype(str)
        .tolist()
    )
    eeg_names = [ch for ch in eeg_names if ch in raw.ch_names]
    exg_names = [f"EXG{i}" for i in range(1, 9) if f"EXG{i}" in raw.ch_names]
    if len(eeg_names) != 64:
        print(f"WARNING: expected 64 EEG channels, found {len(eeg_names)}")
    if len(exg_names) != 8:
        print(f"WARNING: expected 8 EXG channels, found {len(exg_names)}")
    return eeg_names, exg_names


def ica_clean(raw_filtered, eeg_names, exg_names, correlation_threshold=0.30):
    """Remove ICA components correlated with EXG without dropping epochs."""
    cleaned = raw_filtered.copy()
    if not exg_names:
        return cleaned.pick(eeg_names)
    ica = mne.preprocessing.ICA(
        n_components=0.99, method="fastica", random_state=42, max_iter="auto",
    )
    ica.fit(cleaned, picks=eeg_names, verbose=False)
    scores = [
        np.asarray(ica.score_sources(cleaned, target=cleaned.get_data(picks=[name])[0]), dtype=float)
        for name in exg_names
    ]
    excluded = np.flatnonzero(np.max(np.abs(np.vstack(scores)), axis=0) >= correlation_threshold)
    ica.apply(cleaned, exclude=excluded, verbose=False)
    return cleaned.pick(eeg_names)


def load_preprocessed_raws(config):
    # Create matched raw and ICA-cleaned versions from the same filtered data.
    raw = mne.io.read_raw_bdf(
        config.bdf_path,
        preload=True,
        stim_channel="Status",
    )
    events = mne.find_events(
        raw,
        stim_channel="Status",
        shortest_event=1,
        consecutive=True,
        verbose=True,
    )
    eeg_names, exg_names = get_eeg_exg_names(raw, config.channels_tsv)
    old_sfreq = raw.info["sfreq"]
    raw.pick(eeg_names + exg_names)
    raw.resample(TARGET_SFREQ, npad="auto")
    # Keep event sample indices aligned after resampling.
    events = events.copy()
    events[:, 0] = np.rint(events[:, 0] * TARGET_SFREQ / old_sfreq).astype(int)
    events = deduplicate_events(events)
    raw.filter(0.5, 45.0, fir_design="firwin")
    raw_condition = raw.copy().pick(eeg_names)
    cleaned_condition = ica_clean(raw, eeg_names, exg_names)
    return {
        "Raw": raw_condition,
        "ICA_Cleaned": cleaned_condition,
    }, events


def build_window_events(events, sfreq, window_cfg):
    # Label mind-wandering as 1 and focus as 0 at each derived epoch start.
    rows = []
    for sample, _, code in events:
        if code == EVENT_MW_REPORT:
            start = sample + int(round(window_cfg["mw_start_offset"] * sfreq))
            if start >= 0:
                rows.append([start, 0, 1])
        elif code in [EVENT_TRIAL_START, EVENT_START_COUNTING]:
            start = sample + int(round(window_cfg["focus_start_offset"] * sfreq))
            if start >= 0:
                rows.append([start, 0, 0])
    return deduplicate_events(np.array(rows, dtype=int))


def make_epochs(raw, events, window_cfg):
    epoch_events = build_window_events(events, raw.info["sfreq"], window_cfg)
    if len(epoch_events) == 0:
        raise RuntimeError("No valid epoch events found.")
    
    sfreq = raw.info["sfreq"]
    focus_samples = round(5.0 * sfreq)
    mw_samples = round(4.9 * sfreq)
    pad_samples = focus_samples - mw_samples
    
    epochs_data = []
    kept_events = []
    for event in epoch_events:
        start = event[0]
        label = event[2]
        stop = start + mw_samples if label == 1 else start + focus_samples
        if start >= 0 and stop <= raw.n_times:
            data = raw.get_data(start=start, stop=stop)
            if label == 1:
                data = np.pad(data, ((0,0), (0, pad_samples)), mode='edge')
            epochs_data.append(data)
            kept_events.append(event)
            
    if not epochs_data:
        raise RuntimeError("No valid epochs found within bounds.")
        
    return mne.EpochsArray(
        np.asarray(epochs_data), raw.info.copy(), tmin=0.0,
        events=np.asarray(kept_events, dtype=int),
        event_id={"Focus": 0, "MW": 1}, verbose=False
    )


# =============================================================================
# Feature extraction
# =============================================================================

def hjorth_features(x):
    dx = np.diff(x)
    ddx = np.diff(dx)
    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)
    activity = var_x
    mobility = np.sqrt(var_dx / var_x) if var_x > 0 else 0.0
    mobility_dx = np.sqrt(var_ddx / var_dx) if var_dx > 0 else 0.0
    complexity = mobility_dx / mobility if mobility > 0 else 0.0
    return activity, mobility, complexity


def safe_entropy_from_hist(x, bins=32):
    hist, _ = np.histogram(x, bins=bins, density=False)
    p = hist.astype(float)
    p = p / (p.sum() + 1e-12)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


def spectral_features(x, sfreq):
    # Welch PSD supplies total/theta/alpha power, entropy, and dominant frequency.
    freqs, psd = welch(
        x, fs=sfreq,
        nperseg=min(len(x), int(2 * sfreq)),
        noverlap=None,
    )
    mask_total = (freqs >= 0.5) & (freqs <= 45.0)
    mask_theta = (freqs >= 4.0) & (freqs <= 8.0)
    mask_alpha = (freqs >= 8.0) & (freqs <= 12.0)
    total_power = np.trapezoid(psd[mask_total], freqs[mask_total]) if np.any(mask_total) else 0.0
    theta_power = np.trapezoid(psd[mask_theta], freqs[mask_theta]) if np.any(mask_theta) else 0.0
    alpha_power = np.trapezoid(psd[mask_alpha], freqs[mask_alpha]) if np.any(mask_alpha) else 0.0
    psd_total = psd[mask_total]
    freqs_total = freqs[mask_total]
    if len(psd_total) > 0 and psd_total.sum() > 0:
        p = psd_total / psd_total.sum()
        spectral_entropy = -np.sum(p * np.log2(p + 1e-12)) / np.log2(len(p))
        dominant_freq = freqs_total[np.argmax(psd_total)]
    else:
        spectral_entropy = 0.0
        dominant_freq = 0.0
    return total_power, theta_power, alpha_power, spectral_entropy, dominant_freq


def channel_feature_vector(x, sfreq):
    x = np.asarray(x, dtype=float)
    mean = np.mean(x)
    var = np.var(x)
    std = np.std(x)
    rms = np.sqrt(np.mean(x ** 2))
    kurt = kurtosis(x, fisher=True, bias=False)
    skw = skew(x, bias=False)
    activity, mobility, complexity = hjorth_features(x)
    p2p = np.ptp(x)
    total_power, theta_power, alpha_power, spec_entropy, dom_freq = spectral_features(x, sfreq)
    energy = np.sum(x ** 2)
    shannon_entropy = safe_entropy_from_hist(x)
    differential_entropy = 0.5 * np.log(2 * np.pi * np.e * (var + 1e-12))
    return [
        mean, var, std, rms, kurt, skw,
        activity, mobility, complexity, p2p,
        total_power, theta_power, alpha_power, spec_entropy,
        dom_freq, energy, shannon_entropy, differential_entropy,
    ]


def extract_epoch_features(epochs, subject, session, window_name, artifact_condition):
    # Flatten channel-wise features while retaining all provenance and label fields.
    data = epochs.get_data()
    sfreq = epochs.info["sfreq"]
    ch_names = epochs.ch_names
    rows = []
    feature_columns = [
        f"{channel}__{feature}"
        for channel in ch_names
        for feature in FEATURE_NAMES
    ]
    for epoch_idx in range(data.shape[0]):
        y = 1 if epochs.events[epoch_idx, 2] == 1 else 0
        state = "MW" if y == 1 else "Focus"
        feature_values = []
        for ch_idx in range(data.shape[1]):
            feature_values.extend(channel_feature_vector(data[epoch_idx, ch_idx, :], sfreq))
        row = {
            "Subject": subject,
            "Session": session,
            "Subject_Session": f"{subject}_ses-{session}",
            "Window": window_name,
            "ArtifactCondition": artifact_condition,
            "Epoch_Index": epoch_idx + 1,
            "Epoch_Sample": int(epochs.events[epoch_idx, 0]),
            "State": state,
            "Label": y,
        }
        row.update(dict(zip(feature_columns, feature_values)))
        rows.append(row)
    return pd.DataFrame(rows)


# =============================================================================
# Classification and grouped cross-validation
# =============================================================================

METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
CONDITION_COLORS = {"Raw": "#66c2a5", "ICA_Cleaned": "#fc8d62"}


def make_models():
    """Return the two classifiers evaluated with identical grouped CV splits."""
    return {
        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")),
        ]),
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")),
        ]),
    }


def make_group_cv(groups, max_splits=5):
    n_splits = min(max_splits, len(np.unique(groups)))
    if n_splits < 2:
        raise ValueError("At least two groups are required for grouped cross-validation.")
    try:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    except Exception:
        return GroupKFold(n_splits=n_splits)


def decision_scores(model, features):
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(features), dtype=float)
    return np.asarray(model.predict_proba(features)[:, 1], dtype=float)


def evaluate_grouped(df, group_column):
    """Grouped out-of-fold evaluation for SVM and Logistic Regression."""
    metadata = {
        "Subject", "Session", "Subject_Session", "Window", "ArtifactCondition",
        "Epoch_Index", "Epoch_Sample", "State", "Label",
    }
    feature_columns = [column for column in df.columns if column not in metadata]
    X = df[feature_columns]
    y = df["Label"].to_numpy(dtype=int)
    groups = df[group_column].to_numpy()
    if len(np.unique(y)) < 2:
        raise ValueError("Both classes are required for classification.")
    cv = make_group_cv(groups)
    summaries, predictions = [], {}
    for model_name, template in make_models().items():
        oof_prediction = np.full(len(y), -1, dtype=int)
        oof_score = np.full(len(y), np.nan, dtype=float)
        for train_index, test_index in cv.split(X, y, groups):
            if len(np.unique(y[train_index])) < 2:
                continue
            model = clone(template)
            model.fit(X.iloc[train_index], y[train_index])
            oof_prediction[test_index] = model.predict(X.iloc[test_index])
            oof_score[test_index] = decision_scores(model, X.iloc[test_index])
        valid = oof_prediction >= 0
        if not valid.any():
            raise RuntimeError(f"{model_name}: no valid out-of-fold predictions were generated.")
        y_true, y_pred, y_score = y[valid], oof_prediction[valid], oof_score[valid]
        summaries.append({
            "Analysis": "Classification",
            "Model": model_name,
            "ArtifactCondition": df["ArtifactCondition"].iloc[0],
            "N_Epochs": len(y_true),
            "N_Groups": len(np.unique(groups)),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, y_score),
        })
        predictions[model_name] = {"y_true": y_true, "y_pred": y_pred}
    return summaries, predictions


def display_condition(condition):
    return condition.replace("_", "-")


def save_metric_comparison(metrics):
    figure, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)
    for axis, model_name in zip(axes, ("SVM", "Logistic Regression")):
        positions = np.arange(len(METRIC_ORDER))
        width = 0.4
        for offset, condition in [(-width / 2, "Raw"), (width / 2, "ICA_Cleaned")]:
            values = (metrics[(metrics["Model"] == model_name) & (metrics["ArtifactCondition"] == condition)]
                      .set_index("Metric").reindex(METRIC_ORDER)["Score"].to_numpy())
            bars = axis.bar(positions + offset, values, width, color=CONDITION_COLORS[condition], label=display_condition(condition))
            axis.bar_label(bars, labels=[f"{value:.3f}" for value in values], padding=3, fontsize=10)
        axis.set_title(f"{model_name}: raw vs ICA-cleaned EEG classification", pad=12)
        axis.set_xlabel("Metric")
        axis.set_xticks(positions, METRIC_ORDER)
        axis.set_ylim(0, 1.0)
        axis.legend(title="Condition", loc="upper right")
    axes[0].set_ylabel("Score")
    figure.tight_layout()
    figure.savefig(PLOT_DIR / "raw_vs_ica_classification_metrics.png", dpi=200, bbox_inches="tight")
    plt.close(figure)


def save_confusion_matrix(y_true, y_pred, model_name, condition, vmax):
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    figure, axis = plt.subplots(figsize=(10, 8))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, square=True,
                vmin=0, vmax=vmax, xticklabels=["Focused", "MW"],
                yticklabels=["Focused", "MW"], ax=axis)
    axis.set_title(f"{model_name}: {display_condition(condition)} out-of-fold confusion matrix", pad=12)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    figure.tight_layout()
    safe_model = model_name.lower().replace(" ", "_")
    figure.savefig(PLOT_DIR / f"{safe_model}_{condition.lower()}_confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close(figure)

# =============================================================================
# GROUPED CLASSIFICATION
# =============================================================================
print("=" * 80)
print("DATASET 2 - Classification (Raw vs ICA-Cleaned)")
print("=" * 80)
print(f"DATASET_ROOT path used: {DATASET_ROOT}")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Kaggle dataset directory not found: {DATASET_ROOT}")

all_feature_tables, failed = [], []
for subject in SUBJECTS:
    for session in SESSIONS:
        try:
            config = make_session_config(subject, session)
            raws_by_condition, events = load_preprocessed_raws(config)
            for condition, raw_obj in raws_by_condition.items():
                for window_name, window_cfg in WINDOWS.items():
                    epochs = make_epochs(raw_obj, events, window_cfg)
                    all_feature_tables.append(extract_epoch_features(
                        epochs, subject=subject, session=session,
                        window_name=window_name, artifact_condition=condition,
                    ))
                    del epochs
            del raws_by_condition, events
            gc.collect()
        except Exception as error:
            print(f"FAILED {subject} session {session}: {type(error).__name__}: {error}")
            failed.append({"Subject": subject, "Session": session, "Error": str(error)})
if not all_feature_tables:
    raise RuntimeError("No recordings were processed successfully.")
df_all = pd.concat(all_feature_tables, ignore_index=True)
df_all.to_csv(OUTDIR / "all_epoch_features.csv", index=False)
pd.DataFrame(failed).to_csv(OUTDIR / "failed_runs.csv", index=False)

all_summaries, all_predictions = [], {}
for window_name in WINDOWS:
    for condition in ARTIFACT_CONDITIONS:
        subset = df_all[(df_all["Window"] == window_name) & (df_all["ArtifactCondition"] == condition)].copy()
        print(f"Evaluating classification: {condition}")
        summaries, predictions = evaluate_grouped(subset, "Subject")
        all_summaries.extend(summaries)
        for model_name, result in predictions.items():
            all_predictions[(condition, model_name)] = result

metrics_df = pd.DataFrame(all_summaries)
metrics_df.to_csv(OUTDIR / "classification_metrics.csv", index=False)
metric_plot_data = metrics_df.melt(
    id_vars=["Analysis", "Model", "ArtifactCondition", "N_Epochs", "N_Groups"],
    value_vars=METRIC_ORDER, var_name="Metric", value_name="Score",
)
save_metric_comparison(metric_plot_data)
shared_vmax = max(confusion_matrix(result["y_true"], result["y_pred"], labels=[0, 1]).max()
                  for result in all_predictions.values())
for (condition, model_name), result in all_predictions.items():
    save_confusion_matrix(result["y_true"], result["y_pred"], model_name, condition, shared_vmax)

print(f"Saved classification metrics to: {OUTDIR / 'classification_metrics.csv'}")
print(f"Saved plots to: {PLOT_DIR}")

DATASET 2 - Classification (Raw vs ICA-Cleaned)
DATASET_ROOT path used: /kaggle/input/datasets/jvkrishwanth/mwdataset
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
1282 events found on stim channel Status
Event IDs: [10 20 21 30 31 50]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
1261 events found on stim channel Status
Event IDs: [10 20 21 30 31 50]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
1284 events found on stim channel Status
Event IDs: [10 20 21 30 31 50]
Finding events on: Status
Trigger channel Status has a non-zero initi